# BulkFormer embedding → grid search 11 mô hình phân loại

Dùng foundation model **BulkFormer** (Kang et al., *Cell Systems* 2026, 101657; pretrain trên ~500k mẫu bulk RNA-seq
từ GEO/ARCHS4, 20.010 gene mã hoá protein; repo `KangBoming/BulkFormer`) làm **bộ trích xuất đặc trưng đóng băng**
(frozen, không fine-tune backbone), rồi chạy **đúng quy trình grid search** của `clf_gridsearch_fe.ipynb`
(cùng 11 mô hình, lưới tham số, chọn theo macro-F1 trên `val`, báo cáo trên `test`, cùng cột kết quả) để so sánh trực tiếp
với PCA / AE / CHNMF và các phương pháp chọn gene. Các "lớp fine-tuning" ở đây chính là các bộ phân loại
(LR, MLP, SVM, ...) học trên embedding.

- **Đầu vào — `DATA_ROOT`**: `DATA_ROOT/<bo>/fold_{1..5}/{train,val,test}.h5` (dữ liệu gốc, cột = gene symbol + `label`).
  - TCGA-LUAD/LUSC: `log2(TPM+1)`, gene protein-coding → đổi về TPM rồi `ln(TPM+1)` (đúng chuẩn BulkFormer).
  - GSE68465: microarray `log2(MAS5+1)` → **pseudo-TPM**: `2^x − 1`, chuẩn hoá tổng mỗi mẫu = 1e6, rồi `ln(1+·)`.
    Đây là phép xấp xỉ (BulkFormer chỉ học trên RNA-seq) — mục 4 kiểm tra nhanh xem mô hình có "hiểu" đầu vào không.
  - Gene symbol → Ensembl theo `bulkformer_gene_info.csv`; gene BulkFormer không có trong dữ liệu được điền `-10`
    (mask token của BulkFormer), tỉ lệ thiếu truyền vào `mask_prob` như notebook gốc của tác giả.
- **Embedding mẫu**: BulkFormer cho embedding từng gene `[n_gene, dim+3]`; gộp (`AGGREGATES`: mean / max) **chỉ trên các gene
  có đo** → vector `dim+3` chiều (643 với bản 147M). Mỗi cách gộp là một "phương pháp trích xuất": `bulkformer-mean`, `bulkformer-max`.
- **Không rò rỉ dữ liệu**: BulkFormer không fit gì trên dữ liệu của mình (embedding tính độc lập từng mẫu, không dùng nhãn),
  nên tính 1 lần cho mọi mẫu rồi chia theo fold. Embedding **không chuẩn hoá** thêm (giống các notebook FE).
  Lưu ý cho khoá luận: TCGA **không** nằm trong dữ liệu pretrain, nhưng đồ thị gene `G_tcga` được dựng từ tương quan
  biểu hiện trên TCGA (không dùng nhãn giai đoạn).
- **Cần GPU** (Kaggle T4/P100 hoặc Colab T4) và **Internet** (clone repo, tải checkpoint Google Drive + file đồ thị Zenodo).
- **Đầu ra** (`WORK/`):
  - `bulkformer_features/fe_bulkformer-<agg>_<bo>/features/fold_k/{train,val,test}.h5` (cột `BF1..` + `label`, cùng format
    với notebook FE) và `fe_bulkformer-<agg>_<bo>.zip` — dùng lại được với `clf_gridsearch_fe.ipynb` / skill `organizing-fe-results`.
  - `bulkformer_features/extraction_metrics.csv` — thời gian, bộ nhớ GPU, tỉ lệ gene khớp, kết quả kiểm tra tái tạo.
  - `classification/bulkformer_<size>/extraction/results_extraction.csv`, `summary_extraction.csv` — cùng cột với bản FE/FS.

In [ ]:
import sys, subprocess, importlib.util
# tables: backend cho pd.read_hdf | xgboost, lightgbm: Kaggle thuong co san
# torch_geometric (GCNConv), performer-pytorch: kien truc BulkFormer | gdown: tai checkpoint tu Google Drive
_missing = [p for p, m in [("tables", "tables"), ("xgboost", "xgboost"), ("lightgbm", "lightgbm"),
                           ("torch_geometric", "torch_geometric"), ("performer-pytorch", "performer_pytorch"),
                           ("gdown", "gdown")]
            if importlib.util.find_spec(m) is None]
if _missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_missing], check=True)

## 1. Cấu hình

In [ ]:
import os

# ====== CAU HINH (sua tay) ======
IN_KAGGLE = os.path.isdir("/kaggle/working")
WORK = "/kaggle/working" if IN_KAGGLE else "/content"
# Du lieu goc: DATA_ROOT/<bo>/fold_k/{train,val,test}.h5 (Colab: tro vao thu muc tren Drive)
DATA_ROOT = "/kaggle/input/datasets/lfreedom2750/experiment-dataset/h5"
DATASETS = ["gse68465", "tcga_luad", "tcga_lusc"]

# --- BulkFormer ---
MODEL_SIZE = "147M"             # "37M" | "50M" | "93M" | "127M" | "147M"
CKPT_PATH = None                # None = tai tu Google Drive; hoac duong dan .pt da upload (vd Kaggle dataset)
AGGREGATES = ["mean", "max"]    # cach gop embedding gene -> embedding mau ("mean" | "max" | "median")
POOL_OBSERVED_ONLY = True       # chi gop tren gene co do (bo gene dien -10)
RENORM_TPM = True               # chuan hoa lai tong TPM = 1e6 tren cac gene khop voi BulkFormer
BATCH_SIZE = 4                  # giam xuong 2 / 1 neu het bo nho GPU
SKIP_EXISTING = True            # bo qua bo da trich xuat xong (chay lai notebook)
N_RECON_SAMPLES = 8             # so mau dung de kiem tra tai tao (muc 4); 0 = bo qua
RECON_MASK_FRAC = 0.15          # ti le gene co do bi che khi kiem tra tai tao

# --- Phan loai (giong clf_gridsearch_fe.ipynb) ---
KNOWN_DATASETS = ["gse68465", "tcga_luad", "tcga_lusc"]   # hau to ten de tach ten bo
N_FOLDS = 5
SEED = 42
N_JOBS = -1
SELECT_METRIC = "f1_macro"    # do do tren val de chon cau hinh
QUICK = False                 # True: luoi nho de chay thu nhanh
MODELS_TO_RUN = ["knn", "gaussian_nb", "logistic_regression", "decision_tree", "random_forest",
                 "linear_svm", "rbf_svm", "mlp", "xgboost", "lightgbm", "stacking"]
FS_RUNS_TO_RUN = None         # None = tat ca; hoac vd ["fe_bulkformer-mean_gse68465"]

FE_DIR = os.path.join(WORK, "bulkformer_features")         # fe_bulkformer-<agg>_<bo>/features/fold_k/*.h5
BF_DIR = os.path.join(WORK, "BulkFormer")                  # repo + checkpoint + file do thi
OUT_ROOT = os.path.join(WORK, "classification")
FS_GROUP = "extraction"
N_FEATURES_TAG = f"bulkformer_{MODEL_SIZE}"
OUT_DIR = os.path.join(OUT_ROOT, N_FEATURES_TAG, FS_GROUP)
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(FE_DIR, exist_ok=True)
RESULTS_CSV = os.path.join(OUT_DIR, f"results_{FS_GROUP}.csv")
SUMMARY_CSV = os.path.join(OUT_DIR, f"summary_{FS_GROUP}.csv")

for _ds in DATASETS:
    for _s in ["train", "val", "test"]:
        assert os.path.isfile(os.path.join(DATA_ROOT, _ds, "fold_1", f"{_s}.h5")), \
            f"Khong thay {DATA_ROOT}/{_ds}/fold_1/{_s}.h5 -- da Add Input chua?"
print("Du lieu  :", DATA_ROOT, DATASETS)
print("BulkFormer", MODEL_SIZE, "| gop:", AGGREGATES)
print("Embedding:", FE_DIR)
print("Ket qua  :", RESULTS_CSV)
print("So CPU   :", os.cpu_count())

## 2. Tải BulkFormer (code, checkpoint, đồ thị gene)

- Code: clone `github.com/KangBoming/BulkFormer` (`utils/BulkFormer.py`, `utils/BulkFormer_block.py`, `utils/Rope.py`).
- Checkpoint 5 cỡ trên Google Drive (link trong README của repo). Nếu `gdown` báo vượt quota, tải tay file `.pt`,
  upload thành Kaggle dataset rồi đặt `CKPT_PATH`.
- Đồ thị gene `G_tcga.pt`, `G_tcga_weight.pt` và `bulkformer_gene_info.csv` lấy từ bản Zenodo mới nhất (README của tác giả
  yêu cầu dùng bản Zenodo vì file cũ không tương thích). `esm2_feature_concat.pt` không cần: hàm khởi tạo hiện tại không dùng `gene_emb`.
- **Khác code gốc**: tác giả dùng `torch_sparse.SparseTensor` cho phép tích chập đồ thị; gói này thường không có wheel cho bản
  torch mới của Kaggle/Colab, nên ở đây truyền `edge_index` + `edge_weight` vào cùng `GCNConv`. Về toán học là **cùng một phép tính**
  (cùng chiều cạnh, cùng chuẩn hoá `gcn_norm`, cùng trọng số), chỉ tốn thêm bộ nhớ GPU khi chạy.

In [ ]:
import glob, json, time, zipfile, itertools, warnings, urllib.request
from collections import OrderedDict
import numpy as np
import pandas as pd
import torch

warnings.filterwarnings("ignore")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
AMP = DEVICE == "cuda"
if DEVICE == "cpu":
    print("CANH BAO: khong co GPU -- BulkFormer tren CPU se rat cham (bat Accelerator GPU)")
else:
    print("GPU:", torch.cuda.get_device_name(0))

if not os.path.isdir(os.path.join(BF_DIR, "utils")):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/KangBoming/BulkFormer", BF_DIR], check=True)
sys.path.insert(0, BF_DIR)

# Checkpoint (Google Drive, theo README cua BulkFormer)
CKPT_GDRIVE_IDS = {
    "37M": "1qY2qaXfKfDot9EMcOF9gr8T0jghrH7km",
    "50M": "12ZYGYrZIQJyodaVicrJpnY_8_JG-hamK",
    "93M": "1s_3XoMaHiBfxi5C8D3bgwwzNrihafaIk",
    "127M": "1-5AdgIpkm8dOm9tuwOXcqS1sUyGN6vHg",
    "147M": "1UtqN_vCh3669Fs-GU5CTE7F7UnuQCAzN",
}
# Sieu tham so kien truc (model/config.py cua BulkFormer)
MODEL_CONFIGS = {
    "37M": {"dim": 128, "p_repeat": 1},
    "50M": {"dim": 256, "p_repeat": 2},
    "93M": {"dim": 512, "p_repeat": 6},
    "127M": {"dim": 640, "p_repeat": 8},
    "147M": {"dim": 640, "p_repeat": 12},
}
for _c in MODEL_CONFIGS.values():
    _c.update(bins=0, gb_repeat=1, bin_head=12, full_head=8, gene_length=20010)

if CKPT_PATH is None:
    import gdown
    CKPT_PATH = os.path.join(BF_DIR, "model", f"bulkformer_{MODEL_SIZE}.pt")
    if not os.path.isfile(CKPT_PATH):
        gdown.download(id=CKPT_GDRIVE_IDS[MODEL_SIZE], output=CKPT_PATH, quiet=False)
assert os.path.isfile(CKPT_PATH), f"Khong thay checkpoint {CKPT_PATH}"

# File do thi + danh sach gene (Zenodo, ban moi nhat)
ZENODO = "https://zenodo.org/api/records/15744294/files/{}/content"
BF_DATA = os.path.join(BF_DIR, "zenodo")
os.makedirs(BF_DATA, exist_ok=True)
for fn in ["G_tcga.pt", "G_tcga_weight.pt", "bulkformer_gene_info.csv"]:
    dest = os.path.join(BF_DATA, fn)
    if not os.path.isfile(dest):
        urllib.request.urlretrieve(ZENODO.format(fn), dest)
    print(f"{fn:28s} {os.path.getsize(dest) / 1e6:.1f} MB")
print("Checkpoint:", CKPT_PATH, f"({os.path.getsize(CKPT_PATH) / 1e6:.0f} MB)")

In [ ]:
from utils.BulkFormer import BulkFormer
from utils.BulkFormer_block import BulkFormer_block


def _block_forward(self, x, graph):
    # Giong BulkFormer_block.forward goc, nhung graph = (edge_index, edge_weight) thay vi SparseTensor
    edge_index, edge_weight = graph
    x = self.layernorm(x)
    x = x + self.g(x, edge_index, edge_weight)
    return self.f(x)


BulkFormer_block.forward = _block_forward

# Code goc: SparseTensor(row=G[1], col=G[0], value=w).t() -> adj_t co hang = G[0] (dich), cot = G[1] (nguon)
# -> tuong duong edge_index = [nguon; dich] = [G[1]; G[0]]
_G = torch.load(os.path.join(BF_DATA, "G_tcga.pt"), map_location="cpu", weights_only=False)
_W = torch.load(os.path.join(BF_DATA, "G_tcga_weight.pt"), map_location="cpu", weights_only=False)
_G = torch.as_tensor(_G).long()
assert _G.shape[0] == 2, f"G_tcga.pt co shape la {tuple(_G.shape)}"
GRAPH = (torch.stack([_G[1], _G[0]]).to(DEVICE), torch.as_tensor(_W).float().to(DEVICE))
print(f"Do thi gene: {GRAPH[0].shape[1]} canh")

model = BulkFormer(graph=GRAPH, gene_emb=None, **MODEL_CONFIGS[MODEL_SIZE]).to(DEVICE)
_ckpt = torch.load(CKPT_PATH, map_location="cpu", weights_only=False)
_ckpt = _ckpt.get("state_dict", _ckpt) if isinstance(_ckpt, dict) else _ckpt
print(model.load_state_dict(OrderedDict((k[7:] if k.startswith("module.") else k, v) for k, v in _ckpt.items())))
model.eval()
EMB_DIM = MODEL_CONFIGS[MODEL_SIZE]["dim"] + 3   # dim + (mask_prob, expr_mean, nonzero_ratio)
print(f"BulkFormer-{MODEL_SIZE}: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M tham so | embedding {EMB_DIM} chieu")
del _ckpt

## 3. Chuẩn bị đầu vào: gene symbol → Ensembl, log2 → ln(TPM+1)

`fold_1` gồm toàn bộ mẫu (train ∪ val ∪ test là một phân hoạch), nên đọc `fold_1` là đủ để có mọi mẫu của một bộ.
BulkFormer không học gì từ dữ liệu này nên tính embedding một lần cho mọi mẫu, rồi tách theo từng fold ở mục 5.

In [ ]:
STAGE_NAMES = {0: "Normal", 1: "I", 2: "II", 3: "III", 4: "IV"}
SPLITS = ["train", "val", "test"]
MASK_TOKEN = -10.0

gene_info = pd.read_csv(os.path.join(BF_DATA, "bulkformer_gene_info.csv"))
BF_GENES = gene_info["ensg_id"].tolist()                 # thu tu gene = thu tu token cua mo hinh
assert len(BF_GENES) == MODEL_CONFIGS[MODEL_SIZE]["gene_length"], len(BF_GENES)
SYM2ENS = gene_info.drop_duplicates("gene_symbol").set_index("gene_symbol")["ensg_id"]


def load_all_samples(ds):
    # fold_1: train + val + test = toan bo mau cua bo
    dfs = [pd.read_hdf(os.path.join(DATA_ROOT, ds, "fold_1", f"{s}.h5"), key="data") for s in SPLITS]
    df = pd.concat(dfs)
    assert df.index.is_unique, f"{ds}: trung sample_id giua cac split"
    return df.drop(columns="label"), df["label"].astype(int)


def to_bulkformer_input(X_log2):
    # log2(v + 1) -> v -> (pseudo-)TPM tren cac gene khop -> ln(TPM + 1) -> 20.010 cot Ensembl, thieu = -10
    X = X_log2.loc[:, X_log2.columns.isin(SYM2ENS.index)].copy()
    X.columns = SYM2ENS[X.columns].values
    X = X.loc[:, ~X.columns.duplicated()]
    v = np.clip(np.exp2(X.values.astype(np.float64)) - 1.0, 0.0, None)
    if RENORM_TPM:
        v = v / np.maximum(v.sum(axis=1, keepdims=True), 1e-6) * 1e6
    lt = pd.DataFrame(np.log1p(v), index=X.index, columns=X.columns)
    out = lt.reindex(columns=BF_GENES, fill_value=MASK_TOKEN).astype(np.float32)
    observed = np.flatnonzero(np.isin(BF_GENES, X.columns))
    return out.values, observed


INPUTS = {}
for ds in DATASETS:
    X_log2, y = load_all_samples(ds)
    X_in, observed = to_bulkformer_input(X_log2)
    mask_prob = 1.0 - len(observed) / len(BF_GENES)
    INPUTS[ds] = {"X": X_in, "y": y, "observed": observed, "mask_prob": mask_prob, "n_genes_data": X_log2.shape[1]}
    obs_vals = X_in[:, observed]
    print(f"{ds:10s} {X_in.shape[0]} mau | {X_log2.shape[1]} gene -> khop {len(observed)}/{len(BF_GENES)} "
          f"(mask_prob {mask_prob:.3f}) | ln(TPM+1): trung vi {np.median(obs_vals):.2f}, "
          f"p99 {np.percentile(obs_vals, 99):.2f}, ti le 0 {np.mean(obs_vals == 0):.2f} | "
          f"lop {y.map(STAGE_NAMES).value_counts().to_dict()}")
    del X_log2

## 4. Kiểm tra nhanh: BulkFormer có "hiểu" đầu vào không? (tái tạo gene bị che)

Che ngẫu nhiên `RECON_MASK_FRAC` gene **có đo** của vài mẫu (đặt `-10`), cho BulkFormer dự đoán lại (`output_expr=True`),
tính tương quan Pearson giữa giá trị dự đoán và giá trị thật trên các vị trí bị che. Tác giả báo cáo tương quan cao trên RNA-seq;
nếu GSE68465 (microarray, pseudo-TPM) thấp hơn hẳn TCGA thì embedding của bộ đó cần được diễn giải thận trọng.

In [ ]:
from scipy.stats import pearsonr, spearmanr


@torch.no_grad()
def bf_forward(xb, mask_prob, output_expr=False):
    with torch.autocast("cuda", enabled=AMP):
        return model(xb, mask_prob=mask_prob, output_expr=output_expr)


@torch.no_grad()
def recon_check(ds, n=N_RECON_SAMPLES, frac=RECON_MASK_FRAC):
    inp = INPUTS[ds]
    rng = np.random.default_rng(SEED)
    rows = rng.choice(len(inp["X"]), size=min(n, len(inp["X"])), replace=False)
    X = inp["X"][rows].copy()
    hide = np.zeros_like(X, dtype=bool)
    for i in range(len(X)):
        hide[i, rng.choice(inp["observed"], size=int(frac * len(inp["observed"])), replace=False)] = True
    truth = X[hide]
    X[hide] = MASK_TOKEN
    mp = inp["mask_prob"] + frac * len(inp["observed"]) / len(BF_GENES)
    pred = np.vstack([bf_forward(torch.as_tensor(X[i:i + BATCH_SIZE], device=DEVICE), mp, True).float().cpu().numpy()
                      for i in range(0, len(X), BATCH_SIZE)])[hide]
    return {"recon_pearson": float(pearsonr(truth, pred)[0]), "recon_spearman": float(spearmanr(truth, pred)[0])}


RECON = {}
if N_RECON_SAMPLES > 0:
    for ds in DATASETS:
        RECON[ds] = recon_check(ds)
        print(f"{ds:10s} tai tao gene bi che: Pearson {RECON[ds]['recon_pearson']:.3f} | "
              f"Spearman {RECON[ds]['recon_spearman']:.3f}")
    torch.cuda.empty_cache()

## 5. Trích xuất embedding và ghi theo fold

Mỗi batch: BulkFormer → embedding gene `[b, 20010, dim+3]` → gộp trên gene có đo (`POOL_OBSERVED_ONLY`) bằng mỗi cách trong
`AGGREGATES` (một lần forward cho mọi cách gộp). Ghi `fe_bulkformer-<agg>_<bo>/features/fold_k/{train,val,test}.h5` theo
đúng sample_id của từng fold trong `DATA_ROOT`, kèm `config.json` và file zip cùng format với các notebook FE.

In [ ]:
import shutil


@torch.no_grad()
def extract_embeddings(X, mask_prob, observed):
    pool_idx = torch.as_tensor(observed if POOL_OBSERVED_ONLY else np.arange(X.shape[1]), device=DEVICE)
    out = {a: [] for a in AGGREGATES}
    for i in range(0, len(X), BATCH_SIZE):
        h = bf_forward(torch.as_tensor(X[i:i + BATCH_SIZE], device=DEVICE), mask_prob)[:, pool_idx, :].float()
        for a in AGGREGATES:
            pooled = {"mean": lambda t: t.mean(1), "max": lambda t: t.amax(1),
                      "median": lambda t: t.median(1).values}[a](h)
            out[a].append(pooled.cpu().numpy())
        del h
    return {a: np.vstack(v).astype(np.float64) for a, v in out.items()}


def run_dir(agg, ds):
    return os.path.join(FE_DIR, f"fe_bulkformer-{agg}_{ds}")


def is_done(agg, ds):
    return os.path.isfile(os.path.join(run_dir(agg, ds), "features", f"fold_{N_FOLDS}", "test.h5"))


def write_folds(agg, ds, emb_df):
    # emb_df: index = sample_id (moi mau) -> ghi lai dung cac split cua tung fold goc
    root = run_dir(agg, ds)
    names = list(emb_df.columns)
    for r in range(1, N_FOLDS + 1):
        out_dir = os.path.join(root, "features", f"fold_{r}")
        os.makedirs(out_dir, exist_ok=True)
        for s in SPLITS:
            src = pd.read_hdf(os.path.join(DATA_ROOT, ds, f"fold_{r}", f"{s}.h5"), key="data")
            df = emb_df.loc[src.index, names].copy()
            df["label"] = src["label"].values
            df.to_hdf(os.path.join(out_dir, f"{s}.h5"), key="data", mode="w")
    with open(os.path.join(root, "config.json"), "w") as f:
        json.dump({"dataset": ds, "data_dir": os.path.join(DATA_ROOT, ds), "method": f"bulkformer-{agg}",
                   "n_features": len(names), "n_folds": N_FOLDS, "seed": SEED,
                   "params": {"model_size": MODEL_SIZE, "checkpoint": os.path.basename(CKPT_PATH),
                              "aggregate": agg, "pool_observed_only": POOL_OBSERVED_ONLY,
                              "renorm_tpm": RENORM_TPM, "mask_prob": INPUTS[ds]["mask_prob"]}}, f, indent=1)
    shutil.make_archive(os.path.join(WORK, f"fe_bulkformer-{agg}_{ds}"), "zip", root)


metric_rows = []
for ds in DATASETS:
    if SKIP_EXISTING and all(is_done(a, ds) for a in AGGREGATES):
        print(f"{ds}: da co embedding, bo qua")
        continue
    inp = INPUTS[ds]
    if DEVICE == "cuda":
        torch.cuda.reset_peak_memory_stats()
    t0 = time.perf_counter()
    embs = extract_embeddings(inp["X"], inp["mask_prob"], inp["observed"])
    if DEVICE == "cuda":
        torch.cuda.synchronize()
    elapsed = time.perf_counter() - t0
    for agg, E in embs.items():
        emb_df = pd.DataFrame(E, index=inp["y"].index, columns=[f"BF{i}" for i in range(1, E.shape[1] + 1)])
        write_folds(agg, ds, emb_df)
    metric_rows.append({
        "dataset": ds, "model_size": MODEL_SIZE, "n_samples": len(inp["X"]), "n_genes_data": inp["n_genes_data"],
        "n_genes_matched": len(inp["observed"]), "mask_prob": inp["mask_prob"], "emb_dim": EMB_DIM,
        "time_sec": elapsed, "time_per_sample_ms": 1e3 * elapsed / len(inp["X"]),
        "peak_gpu_memory_mb": torch.cuda.max_memory_allocated() / 1024 ** 2 if DEVICE == "cuda" else np.nan,
        **RECON.get(ds, {}),
    })
    print(f"{ds:10s} {len(inp['X'])} mau -> {EMB_DIM} chieu x {len(AGGREGATES)} cach gop | {elapsed:.1f}s "
          f"({1e3 * elapsed / len(inp['X']):.0f} ms/mau) | peak GPU {metric_rows[-1]['peak_gpu_memory_mb']:.0f} MB",
          flush=True)

if metric_rows:
    _mpath = os.path.join(FE_DIR, "extraction_metrics.csv")
    _m = pd.DataFrame(metric_rows)
    if os.path.exists(_mpath):
        _old = pd.read_csv(_mpath)
        _m = pd.concat([_old[~_old.dataset.isin(_m.dataset)], _m], ignore_index=True)
    _m.to_csv(_mpath, index=False)
    display(_m)

# Giai phong GPU truoc khi chay grid search (CPU)
del model, GRAPH, INPUTS
if DEVICE == "cuda":
    torch.cuda.empty_cache()

## 6. Tìm các embedding vừa trích xuất trong `FE_DIR`

In [ ]:
import glob, json, time, zipfile, itertools, warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

STAGE_NAMES = {0: "Normal", 1: "I", 2: "II", 3: "III", 4: "IV"}
SPLITS = ["train", "val", "test"]
EXTRACT_DIR = os.path.join("/kaggle/working/_fe_extracted", N_FEATURES_TAG)


def parse_run_name(entry):
    # "fe_pca_gse68465(.zip)" -> ("fe_pca_gse68465", "pca", "gse68465")
    stem = entry[:-4] if entry.endswith(".zip") else entry
    for ds in sorted(KNOWN_DATASETS, key=len, reverse=True):
        if stem.endswith("_" + ds):
            method = stem[:-len(ds) - 1]
            return stem, (method[3:] if method.startswith("fe_") else method), ds
    return None


def find_fold_dirs(root):
    # fold_k/{train,val,test}.h5 o bat ky do sau nao (features/fold_k trong zip, hoac fold_k truc tiep)
    out = {}
    for r in range(1, N_FOLDS + 1):
        hits = sorted(glob.glob(os.path.join(root, "**", f"fold_{r}", "train.h5"), recursive=True))
        assert hits, f"Khong thay fold_{r}/train.h5 trong {root}"
        d = os.path.dirname(hits[0])
        for s in SPLITS:
            assert os.path.isfile(os.path.join(d, f"{s}.h5")), f"Khong thay {d}/{s}.h5"
        out[r] = d
    return out


FS_RUNS = {}
for entry in sorted(os.listdir(FE_DIR)):   # thu muc "x" dung truoc "x.zip" -> uu tien ban da giai nen
    path = os.path.join(FE_DIR, entry)
    parsed = parse_run_name(entry)
    if parsed is None or parsed[0] in FS_RUNS:
        continue
    stem, method, ds = parsed
    if FS_RUNS_TO_RUN is not None and stem not in FS_RUNS_TO_RUN:
        continue
    if entry.endswith(".zip"):
        root = os.path.join(EXTRACT_DIR, stem)
        if not os.path.isdir(root):
            # chi giai nen features/ (du lieu sau giam chieu) va config.json
            with zipfile.ZipFile(path) as zf:
                zf.extractall(root, members=[m for m in zf.namelist()
                                             if m.startswith("features/") or m.endswith("config.json")])
    elif os.path.isdir(path):
        root = path
    else:
        continue
    FS_RUNS[stem] = {"method": method, "dataset": ds, "folds": find_fold_dirs(root)}

assert FS_RUNS, f"Khong tim thay phuong phap trich xuat nao trong {FE_DIR}"
print(pd.DataFrame([{"fe_run": k, "method": v["method"], "dataset": v["dataset"]}
                    for k, v in FS_RUNS.items()]).to_string(index=False))

## 7. Đọc dữ liệu

In [ ]:
_DATA_CACHE = {}


def load_run(stem):
    # Chi giu du lieu cua 1 fe_run trong RAM (moi phuong phap x bo co file h5 rieng)
    if stem not in _DATA_CACHE:
        _DATA_CACHE.clear()
        _DATA_CACHE[stem] = {r: {s: pd.read_hdf(os.path.join(d, f"{s}.h5"), key="data") for s in SPLITS}
                             for r, d in FS_RUNS[stem]["folds"].items()}
    return _DATA_CACHE[stem]


def get_xy(df, feats):
    missing = [f for f in feats if f not in df.columns]
    assert not missing, f"{len(missing)} dac trung khong co trong du lieu, vd {missing[:5]}"
    # Dac trung da trich xuat (fit tren train o notebook trich xuat), KHONG chuan hoa them
    return df[feats].values.astype(np.float64), df["label"].values.astype(int)


# Kiem tra nhanh: so dac trung cua tung fe_run (fold 1)
for stem, info in FS_RUNS.items():
    cols = pd.read_hdf(os.path.join(info["folds"][1], "train.h5"), key="data").columns
    info["n_features"] = len(cols) - ("label" in cols)
    print(f"{stem:30s} {info['n_features']} dac trung | vd {list(cols[:3])}")

## 8. Mô hình và lưới tham số

Lưới lấy từ `model_tuning_5fold_kaggle.ipynb`, chỉnh lại khoảng `C`/`gamma` cho phù hợp embedding vài trăm chiều
(lưới cũ dò cho ~20.000 gene: `gamma="scale"` giờ cỡ 1e-3 thay vì 1e-5). Cấu hình mặc định luôn được chạy kèm.
XGBoost không có `class_weight` nên `class_weight="balanced"` được thực hiện bằng `sample_weight`.

**Stacking** (theo bài báo): base learner = MLP, RF, KNN, SVM — dùng **cấu hình tốt nhất trên val của
cùng fold** (SVM = linear hoặc RBF, cái nào val macro-F1 cao hơn); meta-learner = XGBoost, học trên dự đoán
out-of-fold (StratifiedKFold trên train) của base learner; chỉ dò lưới tham số của meta-learner.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

SVM_MAX_ITER = 1_000_000   # tranh libsvm chay vo han tren du lieu khong chuan hoa


def build_model(model, params):
    p = dict(params)
    if model == "knn":
        return KNeighborsClassifier(n_jobs=N_JOBS, **p)
    if model == "gaussian_nb":
        return GaussianNB(**p)
    if model == "logistic_regression":
        return LogisticRegression(max_iter=5000, random_state=SEED, **p)
    if model == "decision_tree":
        return DecisionTreeClassifier(random_state=SEED, **p)
    if model == "random_forest":
        return RandomForestClassifier(n_jobs=N_JOBS, random_state=SEED, **p)
    if model == "linear_svm":
        return SVC(kernel="linear", max_iter=SVM_MAX_ITER, random_state=SEED, **p)
    if model == "rbf_svm":
        return SVC(kernel="rbf", max_iter=SVM_MAX_ITER, random_state=SEED, **p)
    if model == "mlp":
        p["hidden_layer_sizes"] = tuple(p["hidden_layer_sizes"])
        return MLPClassifier(max_iter=500, random_state=SEED, **p)
    if model == "xgboost":
        p.pop("class_weight", None)   # xu ly bang sample_weight trong fit_model
        return XGBClassifier(tree_method="hist", n_jobs=N_JOBS, random_state=SEED, verbosity=0, **p)
    if model == "lightgbm":
        return LGBMClassifier(n_jobs=N_JOBS, random_state=SEED, verbose=-1, **p)
    raise ValueError(model)


def fit_model(model, params, X, y):
    est = build_model(model, params)
    kw = {}
    if model == "xgboost" and params.get("class_weight") == "balanced":
        kw["sample_weight"] = compute_sample_weight("balanced", y)
    t0 = time.perf_counter()
    est.fit(X, y, **kw)
    return est, time.perf_counter() - t0


DEFAULTS = {
    "knn": {"n_neighbors": 5, "weights": "uniform", "metric": "minkowski"},
    "gaussian_nb": {"var_smoothing": 1e-9},
    "logistic_regression": {"C": 1.0, "class_weight": None},
    "decision_tree": {"max_depth": None, "min_samples_leaf": 1, "criterion": "gini", "class_weight": None},
    "random_forest": {"n_estimators": 500, "max_features": "sqrt", "min_samples_leaf": 1,
                      "class_weight": None, "max_samples": None},
    "linear_svm": {"C": 1.0, "class_weight": None},
    "rbf_svm": {"C": 1.0, "gamma": "scale", "class_weight": None},
    "mlp": {"hidden_layer_sizes": [100], "alpha": 1e-4},
    "xgboost": {"n_estimators": 100, "max_depth": 6, "learning_rate": 0.3, "subsample": 1.0,
                "colsample_bytree": 1.0, "class_weight": None},
    "lightgbm": {"n_estimators": 100, "num_leaves": 31, "learning_rate": 0.1, "min_child_samples": 20,
                 "class_weight": None},
    # stacking: tham so cua meta-learner XGBoost
    "stacking": {"n_estimators": 100, "max_depth": 6, "learning_rate": 0.3},
}

if QUICK:
    GRIDS = {
        "knn": {"n_neighbors": [5, 11], "weights": ["distance"], "metric": ["minkowski", "cosine"]},
        "gaussian_nb": {"var_smoothing": [1e-9, 1e-3]},
        "logistic_regression": {"C": [0.01, 1.0], "class_weight": ["balanced"]},
        "decision_tree": {"max_depth": [None, 5], "min_samples_leaf": [5], "criterion": ["gini"],
                          "class_weight": ["balanced"]},
        "random_forest": {"n_estimators": [300], "max_features": ["sqrt"], "min_samples_leaf": [1, 5],
                          "class_weight": ["balanced"], "max_samples": [None]},
        "linear_svm": {"C": [1e-3, 1e-1], "class_weight": ["balanced"]},
        "rbf_svm": {"C": [1.0, 10.0], "gamma": ["scale"], "class_weight": ["balanced"]},
        "mlp": {"hidden_layer_sizes": [[100]], "alpha": [1e-4, 1.0]},
        "xgboost": {"n_estimators": [200], "max_depth": [3], "learning_rate": [0.1], "subsample": [0.8],
                    "colsample_bytree": [0.5], "class_weight": [None, "balanced"]},
        "lightgbm": {"n_estimators": [200], "num_leaves": [15], "learning_rate": [0.1],
                     "min_child_samples": [10], "class_weight": [None, "balanced"]},
        "stacking": {"n_estimators": [100], "max_depth": [3], "learning_rate": [0.1]},
    }
else:
    GRIDS = {
        "knn": {"n_neighbors": [3, 5, 7, 11, 15, 21], "weights": ["uniform", "distance"],
                "metric": ["minkowski", "cosine"]},                                    # 24
        "gaussian_nb": {"var_smoothing": [1e-9, 1e-7, 1e-5, 1e-3, 1e-2, 1e-1]},         # 6
        "logistic_regression": {"C": [1e-4, 1e-3, 1e-2, 1e-1, 1.0, 10.0],
                                "class_weight": [None, "balanced"]},                   # 12
        "decision_tree": {"max_depth": [None, 3, 5, 10, 20], "min_samples_leaf": [1, 5, 10],
                          "criterion": ["gini", "entropy"],
                          "class_weight": [None, "balanced"]},                         # 60
        "random_forest": {"n_estimators": [500], "max_features": ["sqrt", "log2", 0.3],
                          "min_samples_leaf": [1, 5, 10], "class_weight": ["balanced"],
                          "max_samples": [None, 0.5]},                                 # 18
        "linear_svm": {"C": [1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1.0],
                       "class_weight": [None, "balanced"]},                            # 12
        "rbf_svm": {"C": [0.1, 1.0, 10.0, 100.0], "gamma": ["scale", 1e-4, 1e-3, 1e-2],
                    "class_weight": [None, "balanced"]},                               # 32
        "mlp": {"hidden_layer_sizes": [[64], [256], [256, 64]],
                "alpha": [1e-4, 1e-2, 1.0, 10.0]},                                     # 12
        "xgboost": {"n_estimators": [200, 500], "max_depth": [3, 6], "learning_rate": [0.05, 0.1],
                    "subsample": [0.8], "colsample_bytree": [0.5, 1.0],
                    "class_weight": [None, "balanced"]},                               # 32
        "lightgbm": {"n_estimators": [200, 500], "num_leaves": [7, 15], "learning_rate": [0.05, 0.1],
                     "min_child_samples": [5, 20], "class_weight": [None, "balanced"]}, # 32
        "stacking": {"n_estimators": [100, 300], "max_depth": [2, 3],
                     "learning_rate": [0.05, 0.1]},                                    # 8
    }


def configs_of(model):
    grid = GRIDS[model]
    cfgs = [dict(zip(grid, v)) for v in itertools.product(*grid.values())]
    return cfgs if DEFAULTS[model] in cfgs else [DEFAULTS[model]] + cfgs


CONFIGS = {m: configs_of(m) for m in MODELS_TO_RUN}
n_cfg = sum(len(c) for c in CONFIGS.values())
print(pd.Series({m: len(c) for m, c in CONFIGS.items()}).to_string())
print(f"Tong: {n_cfg} cau hinh x {N_FOLDS} fold x {len(FS_RUNS)} (phuong phap trich xuat x bo) "
      f"= {n_cfg * N_FOLDS * len(FS_RUNS)} lan huan luyen (chua tinh CV noi bo cua stacking)")

### Stacking: base MLP, RF, KNN, SVM → meta XGBoost

In [ ]:
STACK_BASES = ["mlp", "random_forest", "knn", "svm"]   # "svm" = linear_svm / rbf_svm tot hon tren val
STACK_CV = 5


def stack_method(model):
    return "decision_function" if model in ("linear_svm", "rbf_svm") else "predict_proba"


def base_output(est, model, X):
    return getattr(est, stack_method(model))(X).reshape(len(X), -1)


class StackedModel:
    # base learner da fit tren toan bo train + meta XGBoost fit tren du doan out-of-fold
    def __init__(self, bases, meta):
        self.bases, self.meta = bases, meta

    def _meta_features(self, X):
        return np.hstack([base_output(est, m, X) for m, est in self.bases])

    def predict(self, X):
        return self.meta.predict(self._meta_features(X))

    def predict_proba(self, X):
        return self.meta.predict_proba(self._meta_features(X))


def fit_stack_bases(bases, X, y):
    # bases: [(model, params)] -> (base da fit, ma tran out-of-fold cho meta, thoi gian)
    n_splits = max(2, min(STACK_CV, np.bincount(y).min()))   # lop hiem (vd LUSC stage IV ~5 mau)
    cv = StratifiedKFold(n_splits, shuffle=True, random_state=SEED)
    t0 = time.perf_counter()
    oof, fitted = [], []
    for model, params in bases:
        oof.append(cross_val_predict(build_model(model, params), X, y, cv=cv,
                                     method=stack_method(model)).reshape(len(y), -1))
        fitted.append((model, build_model(model, params).fit(X, y)))
    return fitted, np.hstack(oof), time.perf_counter() - t0


def fit_stacking(stack_state, meta_params, y):
    fitted, oof, base_s = stack_state
    meta, meta_s = fit_model("xgboost", meta_params, oof, y)
    return StackedModel(fitted, meta), base_s + meta_s

## 9. Độ đo

In [ ]:
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, precision_score, recall_score,
                             f1_score, matthews_corrcoef, cohen_kappa_score, roc_auc_score,
                             confusion_matrix)

METRIC_NAMES = ["accuracy", "balanced_accuracy", "precision_macro", "precision_weighted",
                "recall_macro", "recall_weighted", "f1_macro", "f1_weighted", "mcc",
                "cohen_kappa", "roc_auc"]


def get_scores(est, X):
    # xac suat (hoac decision_function voi SVC) de tinh ROC-AUC; cot j <-> lop ma hoa j
    if hasattr(est, "predict_proba"):
        return est.predict_proba(X)
    d = est.decision_function(X)
    return np.column_stack([-d, d]) if d.ndim == 1 else d


def roc_auc_ovr_macro(y, scores):
    # one-vs-rest, trung binh macro; bo qua lop khong co mat trong y (AUC khong xac dinh)
    aucs = [roc_auc_score(y == c, scores[:, c]) for c in range(scores.shape[1])
            if 0 < (y == c).sum() < len(y)]
    return float(np.mean(aucs)) if aucs else np.nan


def compute_metrics(y, p, scores):
    kw = dict(zero_division=0)
    return {
        "accuracy": accuracy_score(y, p),
        "balanced_accuracy": balanced_accuracy_score(y, p),
        "precision_macro": precision_score(y, p, average="macro", **kw),
        "precision_weighted": precision_score(y, p, average="weighted", **kw),
        "recall_macro": recall_score(y, p, average="macro", **kw),
        "recall_weighted": recall_score(y, p, average="weighted", **kw),
        "f1_macro": f1_score(y, p, average="macro", **kw),
        "f1_weighted": f1_score(y, p, average="weighted", **kw),
        "mcc": matthews_corrcoef(y, p),
        "cohen_kappa": cohen_kappa_score(y, p),
        "roc_auc": roc_auc_ovr_macro(y, scores),
    }


# Do do chon cau hinh tren val (chi dung nhan du doan)
SCORERS = {
    "f1_macro": lambda y, p: f1_score(y, p, average="macro", zero_division=0),
    "balanced_accuracy": balanced_accuracy_score,
    "mcc": matthews_corrcoef,
    "accuracy": accuracy_score,
}
select_score = SCORERS[SELECT_METRIC]


def evaluate(est, X, y):
    t0 = time.perf_counter()
    p = est.predict(X)
    infer_s = time.perf_counter() - t0
    return compute_metrics(y, p, get_scores(est, X)), infer_s, p

## 10. Grid search trên 5 fold

Mỗi (phương pháp trích xuất × bộ, fold, mô hình): thử mọi cấu hình, fit trên `train`, chọn theo `SELECT_METRIC` trên `val`,
rồi đánh giá cấu hình tốt nhất trên `val` và `test`. `train_time_s` = thời gian fit cấu hình tốt nhất
(stacking: gồm CV out-of-fold + fit base + fit meta), `infer_time_test_s` = thời gian `predict` trên test,
`tuning_time_s` = tổng thời gian dò lưới của mô hình đó. Mỗi mô hình xong được ghi ngay vào `RESULTS_CSV`.

In [ ]:
def grid_search(fit_fn, configs, X_va, y_va):
    best, t0 = None, time.perf_counter()
    for params in configs:
        est, fit_s = fit_fn(params)
        score = select_score(y_va, est.predict(X_va))
        if best is None or score > best["score"]:
            best = {"params": params, "est": est, "fit_s": fit_s, "score": score}
    best["tuning_s"] = time.perf_counter() - t0
    return best


def make_row(stem, info, r, model, best, n_configs, sets, classes, feats):
    (X_tr, y_tr), (X_va, y_va), (X_te, y_te) = sets
    m_va, _, _ = evaluate(best["est"], X_va, y_va)
    m_te, infer_s, p_te = evaluate(best["est"], X_te, y_te)
    cm = confusion_matrix(y_te, p_te, labels=np.arange(len(classes)))
    return {
        "n_features": N_FEATURES_TAG, "fs_group": FS_GROUP, "fs_method": info["method"], "fs_run": stem,
        "dataset": info["dataset"], "fold": r, "model": model, "n_genes": len(feats),   # = so dac trung trich xuat (giu ten cot cua ban FS)
        "n_train": len(y_tr), "n_val": len(y_va), "n_test": len(y_te),
        "best_params": json.dumps(best["params"]), "n_configs": n_configs,
        "train_time_s": best["fit_s"], "tuning_time_s": best["tuning_s"],
        "infer_time_test_s": infer_s, "infer_time_per_sample_ms": 1e3 * infer_s / len(y_te),
        **{f"val_{k}": v for k, v in m_va.items()},
        **{f"test_{k}": v for k, v in m_te.items()},
        "class_labels": json.dumps([STAGE_NAMES.get(c, str(c)) for c in classes]),
        "confusion_matrix_test": json.dumps(cm.tolist()),
    }


def append_row(row):
    pd.DataFrame([row]).to_csv(RESULTS_CSV, mode="a", index=False, header=not os.path.exists(RESULTS_CSV))

In [ ]:
prev = pd.read_csv(RESULTS_CSV) if os.path.exists(RESULTS_CSV) else pd.DataFrame(
    columns=["fs_run", "fold", "model", "best_params", f"val_{SELECT_METRIC}"])
done = set(zip(prev["fs_run"], prev["fold"].astype(int), prev["model"]))
if done:
    print(f"Tiep tuc: da co {len(done)} dong trong {RESULTS_CSV}")

t_start = time.time()
for stem, info in sorted(FS_RUNS.items(), key=lambda kv: (kv[1]["dataset"], kv[0])):
    for r in range(1, N_FOLDS + 1):
        todo = [m for m in MODELS_TO_RUN if (stem, r, m) not in done]
        if not todo:
            continue
        fold = load_run(stem)[r]
        feats = [c for c in fold["train"].columns if c != "label"]   # toan bo dac trung trich xuat
        (X_tr, y_tr), (X_va, y_va), (X_te, y_te) = [get_xy(fold[s], feats) for s in SPLITS]

        # Ma hoa nhan ve 0..K-1 theo cac lop cua train (XGBoost yeu cau; GSE68465 khong co stage IV)
        classes = np.unique(y_tr)
        assert set(y_va) <= set(classes) and set(y_te) <= set(classes), f"{stem} fold {r}: val/test co lop la"
        y_tr, y_va, y_te = [np.searchsorted(classes, y) for y in (y_tr, y_va, y_te)]
        sets = [(X_tr, y_tr), (X_va, y_va), (X_te, y_te)]

        # cau hinh tot nhat cua fold nay (tu lan chay truoc neu co) -> base learner cho stacking
        fold_best = {row.model: (json.loads(row.best_params), getattr(row, f"val_{SELECT_METRIC}"))
                     for row in prev[(prev.fs_run == stem) & (prev.fold == r)].itertuples()}

        for model in [m for m in todo if m != "stacking"] + [m for m in todo if m == "stacking"]:
            t0 = time.time()
            if model == "stacking":
                svm = max(["linear_svm", "rbf_svm"], key=lambda m: fold_best.get(m, (None, -1))[1])
                bases = []
                for b in STACK_BASES:
                    b = svm if b == "svm" else b
                    if b not in fold_best:
                        print(f"  [stacking] chua co cau hinh tot nhat cua {b}, dung DEFAULTS")
                    bases.append((b, fold_best.get(b, (DEFAULTS[b], None))[0]))
                state = fit_stack_bases(bases, X_tr, y_tr)
                best = grid_search(lambda p: fit_stacking(state, p, y_tr), CONFIGS[model], X_va, y_va)
                best["params"] = {"meta_xgboost": best["params"], "bases": {b: p for b, p in bases}}
            else:
                best = grid_search(lambda p: fit_model(model, p, X_tr, y_tr), CONFIGS[model], X_va, y_va)
            row = make_row(stem, info, r, model, best, len(CONFIGS[model]), sets, classes, feats)
            append_row(row)
            fold_best[model] = (best["params"], row[f"val_{SELECT_METRIC}"])
            done.add((stem, r, model))
            print(f"{stem} | fold {r} | {model:20s} val F1 {row['val_f1_macro']:.3f} | "
                  f"test F1 {row['test_f1_macro']:.3f} acc {row['test_accuracy']:.3f} "
                  f"AUC {row['test_roc_auc']:.3f} | {time.time() - t0:.0f}s "
                  f"(tong {(time.time() - t_start) / 60:.1f} phut)", flush=True)

print("Xong grid search:", RESULTS_CSV)

## 11. Tổng hợp: trung bình qua 5 fold

`summary_<nhom>.csv`: mỗi dòng = (phương pháp trích xuất, bộ, mô hình); cột độ đo = trung bình 5 fold,
cột `*_std` = độ lệch chuẩn của độ đo test, `confusion_matrix_test_sum` = cộng confusion matrix test
của 5 fold (mỗi mẫu làm test đúng 1 lần).

In [ ]:
res = pd.read_csv(RESULTS_CSV)
KEYS = ["n_features", "fs_group", "fs_method", "dataset", "model"]
TIME_COLS = ["train_time_s", "tuning_time_s", "infer_time_test_s", "infer_time_per_sample_ms"]
VAL_COLS = [f"val_{m}" for m in METRIC_NAMES]
TEST_COLS = [f"test_{m}" for m in METRIC_NAMES]


def sum_cm(s):
    mats = [np.array(json.loads(x)) for x in s]
    return json.dumps(np.sum(mats, axis=0).tolist()) if len({m.shape for m in mats}) == 1 else ""


g = res.groupby(KEYS, sort=False)
summary = g[TEST_COLS].mean()
for c in TEST_COLS:
    summary[f"{c}_std"] = g[c].std()
summary = summary[[c for t in TEST_COLS for c in (t, f"{t}_std")]]
summary = summary.join(g[VAL_COLS + TIME_COLS].mean())
summary["n_folds"] = g.size()
summary["class_labels"] = g["class_labels"].first()
summary["confusion_matrix_test_sum"] = g["confusion_matrix_test"].apply(sum_cm)
summary = summary.reset_index()
summary.to_csv(SUMMARY_CSV, index=False)
print("Da luu:", SUMMARY_CSV, summary.shape)

# Bang nhanh: test macro-F1 trung binh, hang = (phuong phap trich xuat, bo), cot = mo hinh
display(summary.pivot_table(index=["fs_method", "dataset"], columns="model", values="test_f1_macro")
        .reindex(columns=[m for m in MODELS_TO_RUN if m in set(summary.model)]).round(3))

In [ ]:
import shutil
zip_out = shutil.make_archive(os.path.join("/kaggle/working", f"classification_{N_FEATURES_TAG}_{FS_GROUP}"),
                              "zip", OUT_DIR)
print("Da nen:", zip_out, f"({os.path.getsize(zip_out) / 1e6:.2f} MB)")